In [0]:

# ============================================================
# NOTEBOOK   : silver_sellers
# PURPOSE    : Clean and transform bronze.sellers table
# SOURCE     : fincompliance_catalog.bronze.sellers
# DESTINATION: fincompliance_catalog.silver.sellers
# TRANSFORMATIONS:
#   - Deduplicate on seller_id
#   - Trim and lowercase city names
#   - Uppercase state codes
#   - Handle null values
#   - Add seller_state_name
#   - Add seller_region
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    when,
    lit,
    current_timestamp
)

In [0]:
# ============================================================
# CELL 5 - READ FROM BRONZE
# ============================================================

df_sellers = spark.table(f"{BRONZE_DB}.sellers")

print("=" * 60)
print("BRONZE.SELLERS — Before Transformation")
print("=" * 60)
print(f"Total rows    : {df_sellers.count():,}")
print(f"Total columns : {len(df_sellers.columns)}")

print("\nColumn names and data types:")
for field in df_sellers.schema.fields:
    print(f"  {field.name:<40} : {field.dataType}")

print("\nSample data BEFORE transformation (3 rows):")
df_sellers.show(3, truncate=True)

print("\nNull counts per column:")
for c in df_sellers.columns:
    null_count = df_sellers \
        .filter(col(c).isNull()) \
        .count()
    if null_count == 0:
        print(f"  {c:<40} : {null_count:,} nulls")
    else:
        print(f"  {c:<40} : {null_count:,} nulls")

print("\nState breakdown:")
df_sellers \
    .groupBy("seller_state") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10, truncate=False)

In [0]:
# ============================================================
# DEDUPLICATION
# ============================================================
rows_before = df_sellers.count()
df_sellers_silver = df_sellers.dropDuplicates(["seller_id"])
rows_after = df_sellers_silver.count()
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Duplicates  : {rows_before - rows_after:,}")

In [0]:
# ============================================================
# TRIM AND LOWERCASE CITY NAMES
# ============================================================
df_sellers_silver = df_sellers_silver \
    .withColumn(
        "seller_city",
        trim(lower(col("seller_city")))
    )

print("City names after standardisation:")
df_sellers_silver.select("seller_city").show(5, truncate=False)
print("Done")

In [0]:
# ============================================================
# UPPERCASE STATE CODES
# ============================================================
df_sellers_silver = df_sellers_silver \
    .withColumn(
        "seller_state",
        upper(trim(col("seller_state")))
    )

print("Unique states:")
df_sellers_silver \
    .select("seller_state") \
    .distinct() \
    .orderBy("seller_state") \
    .show(30, truncate=False)
print("Done")

In [0]:
# ============================================================
# HANDLE NULL CITY AND STATE
# ============================================================
df_sellers_silver = df_sellers_silver \
    .withColumn(
        "seller_city",
        when(col("seller_city").isNull(), lit("unknown"))
        .otherwise(col("seller_city"))
    ) \
    .withColumn(
        "seller_state",
        when(
            col("seller_state").isNull() |
            (col("seller_state") == ""),
            lit("unknown")
        )
        .otherwise(col("seller_state"))
    )

print("Null handling complete")
print(f"Total rows : {df_sellers_silver.count():,}")

In [0]:
# ============================================================
# ADD SELLER STATE NAME
# ============================================================
df_sellers_silver = df_sellers_silver \
    .withColumn(
        "seller_state_name",
        when(col("seller_state") == "SP", "Sao Paulo")
        .when(col("seller_state") == "RJ", "Rio de Janeiro")
        .when(col("seller_state") == "MG", "Minas Gerais")
        .when(col("seller_state") == "RS", "Rio Grande do Sul")
        .when(col("seller_state") == "PR", "Parana")
        .when(col("seller_state") == "SC", "Santa Catarina")
        .when(col("seller_state") == "BA", "Bahia")
        .when(col("seller_state") == "GO", "Goias")
        .when(col("seller_state") == "ES", "Espirito Santo")
        .when(col("seller_state") == "PE", "Pernambuco")
        .when(col("seller_state") == "CE", "Ceara")
        .when(col("seller_state") == "MA", "Maranhao")
        .when(col("seller_state") == "MT", "Mato Grosso")
        .when(col("seller_state") == "MS", "Mato Grosso do Sul")
        .when(col("seller_state") == "PA", "Para")
        .when(col("seller_state") == "RN", "Rio Grande do Norte")
        .when(col("seller_state") == "PB", "Paraiba")
        .when(col("seller_state") == "SE", "Sergipe")
        .when(col("seller_state") == "AL", "Alagoas")
        .when(col("seller_state") == "PI", "Piaui")
        .when(col("seller_state") == "AM", "Amazonas")
        .when(col("seller_state") == "AC", "Acre")
        .when(col("seller_state") == "RO", "Rondonia")
        .when(col("seller_state") == "RR", "Roraima")
        .when(col("seller_state") == "AP", "Amapa")
        .when(col("seller_state") == "TO", "Tocantins")
        .when(col("seller_state") == "DF", "Distrito Federal")
        .otherwise("Unknown")
    )

print("State name mapping complete")
df_sellers_silver \
    .select("seller_state", "seller_state_name") \
    .distinct() \
    .orderBy("seller_state") \
    .show(30, truncate=False)

In [0]:
# ============================================================
# ADD SELLER REGION
# ============================================================
df_sellers_silver = df_sellers_silver \
    .withColumn(
        "seller_region",
        when(col("seller_state").isin(
            ["SP", "RJ", "MG", "ES"]), "Southeast")
        .when(col("seller_state").isin(
            ["RS", "SC", "PR"]), "South")
        .when(col("seller_state").isin(
            ["BA", "PE", "CE", "MA", "PI",
             "RN", "PB", "SE", "AL"]), "Northeast")
        .when(col("seller_state").isin(
            ["AM", "PA", "AC", "RO",
             "RR", "AP", "TO"]), "North")
        .when(col("seller_state").isin(
            ["GO", "MT", "MS", "DF"]), "Central-West")
        .otherwise("Unknown")
    )

print("Seller count by region:")
df_sellers_silver \
    .groupBy("seller_region") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

In [0]:
# ============================================================
# DROP BRONZE AUDIT COLUMNS AND ADD SILVER AUDIT COLUMN
# ============================================================
df_sellers_silver = df_sellers_silver \
    .drop("ingestion_timestamp", "source_file") \
    .withColumn("silver_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_sellers_silver.columns:
    print(f"  {col_name}")
print(f"Total rows : {df_sellers_silver.count():,}")

In [0]:
# ============================================================
# WRITE TO SILVER
# ============================================================
(
    df_sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.sellers")
)

print(f"Written to {SILVER_DB}.sellers")
print(f"Rows : {df_sellers_silver.count():,}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("SILVER.SELLERS VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.sellers")
bronze_count = spark.table(f"{BRONZE_DB}.sellers").count()
silver_count = df_silver.count()

print(f"Bronze rows : {bronze_count:,}")
print(f"Silver rows : {silver_count:,}")
if bronze_count == silver_count:
    print("Row counts match")

print("\nRegion breakdown:")
df_silver.groupBy("seller_region").count() \
    .orderBy("count", ascending=False).show(truncate=False)

print("=" * 60)
print("silver.sellers verification complete!")
print(f"Cols : {len(df_silver.columns)}")
print("=" * 60)